# RQ1: XAI Explanation Stability Benchmark (Multi-Budget Analysis)
Evaluation of token attribution robustness against meaning-preserving prompt perturbations across computational budgets (Integrated Gradients, LIME, SHAP, Occlusion, Grad×Input).

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')
DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# 2. GitHub Configuration (for private repo provide PAT token or add GH_TOKEN in Secrets)
GITHUB_TOKEN = "" #@param {type:"string"}
REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

try:
    from google.colab import userdata
    token_from_secrets = userdata.get('GH_TOKEN')
    if token_from_secrets:
        GITHUB_TOKEN = token_from_secrets
except Exception:
    pass

if GITHUB_TOKEN.strip():
    CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

REPO_DIR = "/content/llm-xai"

if not os.path.exists(REPO_DIR):
    res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
    print("\n".join(res))
    if not os.path.exists(REPO_DIR):
        raise RuntimeError("Cloning error! If repository is private, provide Personal Access Token (PAT) in GITHUB_TOKEN or set GH_TOKEN secret.")
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

# 3. Dependencies (upgrade numpy and packages to prevent StringDType collision)
!pip install -q --upgrade numpy "transformers>=4.43.0,<4.47.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.46.0" scikit-learn scipy pandas matplotlib huggingface_hub

# 4. HuggingFace Token (for gated models)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

### 3. Pipeline Smoke Test (2 samples)

In [ ]:
auto_shutdown = False #@param {type:"boolean"}

!python scripts/run_rq1_grid.py \
    --model microsoft/phi-2 \
    --samples 2 \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --ig-grid 5 \
    --lime-grid 50 \
    --shap-grid 50 \
    --output-dir /content/smoke_test \
    --no-resume

if auto_shutdown:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()


### 4. Main RQ1 Experiment (Full Hyperparameter Grid)

In [ ]:
model = "microsoft/phi-2" #@param ["microsoft/phi-2", "meta-llama/Llama-3.1-8B", "Qwen/Qwen2.5-7B"]
samples = 50 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

ig_grid = "10,20,50,100,200,300" #@param {type:"string"}
lime_grid = "100,250,500,1000,1500,2500,3000" #@param {type:"string"}
shap_grid = "100,250,500,1000,1500,2500,3000" #@param {type:"string"}

max_run_samples = 0 #@param {type:"integer"}
no_noise_floor = False #@param {type:"boolean"}
auto_shutdown = False #@param {type:"boolean"}

tag = model.split("/")[-1]
output_dir = f"{DRIVE_RESULTS}/rq1_grid_{tag}"
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
noise_flag = "--no-noise-floor" if no_noise_floor else ""

!python scripts/run_rq1_grid.py \
    --model {model} \
    --samples {samples} \
    --seed {seed} \
    --methods {methods} \
    --ig-grid {ig_grid} \
    --lime-grid {lime_grid} \
    --shap-grid {shap_grid} \
    {limit_flag} \
    {noise_flag} \
    --output-dir {output_dir} \
    --no-resume

if auto_shutdown:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()


### 5. Results Summary Table

In [ ]:
import os
import pandas as pd
from IPython.display import display

summary_csv = os.path.join(output_dir, "rq1_grid_summary.csv")
if os.path.exists(summary_csv):
    df_summary = pd.read_csv(summary_csv)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    display(df_summary.round(3))
else:
    print(f"Summary file {summary_csv} does not exist.")

### 6. Stability vs. Computational Cost (Pareto Frontier)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

summary_csv = os.path.join(output_dir, "rq1_grid_summary.csv")
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    
    plt.figure(figsize=(9, 5.5), dpi=150)
    colors = {
        "IntegratedGradients": "#1f77b4",
        "LIME": "#2ca02c",
        "SHAP": "#d62728",
        "Occlusion": "#ff7f0e",
        "GradientXInput": "#9467bd"
    }
    
    for family, group in df.groupby("method_family"):
        group = group.sort_values("forward_passes_mean")
        plt.plot(group["forward_passes_mean"], group["spearman_mean"], marker="o", label=family, color=colors.get(family, "black"), linewidth=2)
        for _, row in group.iterrows():
            if pd.notna(row["param_value"]):
                plt.annotate(f"{int(row['param_value'])}", (row["forward_passes_mean"], row["spearman_mean"]), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
                
    plt.xscale("log")
    plt.xlabel("Mean Forward Pass Operations (log scale)")
    plt.ylabel(r"Stability (Spearman $\rho$)")
    plt.title(f"RQ1: Stability vs. Computational Cost – {tag}", fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(title="Method")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "rq1_pareto_curve.png"))
    plt.show()
else:
    print("No summary file found.")

### 7. Automatic Session Teardown (Stop Compute Units Billing)
Run this cell (or leave at the end of the 'Run all' queue for overnight execution) to automatically release the GPU and terminate billing immediately after figures are saved.

In [ ]:
from google.colab import runtime
print("All analyses and figures generated! Disconnecting Colab runtime...")
runtime.unassign()
